In [38]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/VNIR_databases/soil/plsda/soil_vnir.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '400':'2498']


data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '400':'2498'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '400':'2498'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

# vamos converter os espectros de reflectância para absorbância
#Xcalclass = np.log10(1 / Xcalclass)
#Xpredclass = np.log10(1 / Xpredclass)

# savitzky-golay smoothing on the vnir spectra
from scipy.signal import savgol_filter
# Xcalclass_prep = pd.DataFrame(savgol_filter(Xcalclass, window_length=11, polyorder=3, deriv=0, axis=1), columns=Xcalclass.columns)
# Xpredclass_prep = pd.DataFrame(savgol_filter(Xpredclass, window_length=11, polyorder=3, deriv=0, axis=1), columns=Xpredclass.columns)
Xcalclass_prep, mean_calclass  = prepr.mc(Xcalclass)
Xpredclass_prep = Xpredclass - mean_calclass

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=4,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

2026-01-22 16:02:08,423 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-22 16:02:08,429 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-22 16:02:08,501 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-22 16:02:08,504 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [39]:
import permutation as perm
spectral_cuts = [(str(start), start, start + 20) for start in range(400, 2500, 20)]
# Função para calcular importância de regiões espectrais por perturbação

# Aplicando a função aos dados
perturbation_importance_df = perm.spectral_perturbation_importance(
    model=pls_model,
    X=Xcalclass_prep,
    y_pred_original=y_pred_cont.values,
    spectral_cuts=spectral_cuts,
    perturbation_value=np.std(y_pred_cont)*(100),  # valor de perturbação
    metric='mean_diff'  # escolha: 'mean_abs_diff', 'mean_diff', 'mean_relative_dev'
 )

# Visualizando os resultados
perturbation_importance_df

,Zone,Start,End,Importance,Abs_Importance,N_Features
0,720,720,740,11.727946,11.727946,11
1,700,700,720,11.481680,11.481680,11
2,740,740,760,11.306711,11.306711,11
3,680,680,700,10.609007,10.609007,11
4,760,760,780,10.374556,10.374556,11
...,...,...,...,...,...,...
100,2020,2020,2040,0.866445,0.866445,11
101,1160,1160,1180,-0.726349,0.726349,11
102,2140,2140,2160,0.177111,0.177111,11
103,1140,1140,1160,0.135413,0.135413,11


# **VIP and SHAP**

In [35]:
import numpy as np
import pandas as pd

# vip
vip_scores_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'VIP_Score' : plsda_results[4].T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_vip = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in vip_scores_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_vip[i] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# reg vet
reg_vet = pd.DataFrame(plsda_results[3].coef_, columns=plsda_results[3].feature_names_in_) # creating a DataFrame with regression coefficients
reg_vet = reg_vet.T
reg_vet.insert(0, 'energy', reg_vet.index) # adding energy column
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy', 'Reg_coef'] # renaming
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs() # adding absolute value column
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True) # sorting by absolute value

# gerando uma nova coluna em reg_vet com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_reg = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
    for i in reg_vet['energy']: # iterando sobre cada valor de energia no reg_vet
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_reg[i] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_reg
reg_vet

# vamos filtrar reg_vet para manter apenas as zonas espectrais únicas com maior valor absoluto do coeficiente de regressão
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
shap_unique_df = pd.read_csv('shap_soil_vnir.csv', sep=';') # loading previously saved shap_unique_df

In [36]:
vip_scores_unique_df

,energy,VIP_Score,Zone
0,732,2.124840,720
1,740,2.115541,740
2,718,2.097244,700
3,760,2.023045,760
4,698,1.982863,680
...,...,...,...
100,1860,0.555505,1860
101,1258,0.545316,1240
102,1180,0.528402,1180
103,1238,0.517629,1220


In [22]:
# vamos utilizar a permutation do sklearn para realizar uma permutacao no modelo plsda
from sklearn.inspection import permutation_importance
perm_importance = permutation_importance(pls_model, 
                                           Xcalclass_prep, 
                                           y_pred_cont,
                                           n_repeats=5,
                                           random_state=0,
                                           scoring='neg_mean_absolute_error',
                                           n_jobs=30)
# pegando as importancias
perm_importances_values = pd.Series(perm_importance.importances_mean, index=Xcalclass_prep.columns)

# vip
perm_df = pd.DataFrame({
    'energy' : plsda_results[4].T.index,
    'Perm' : perm_importance.importances_mean
})
perm_df = perm_df.sort_values(by='Perm', ascending=False).reset_index(drop=True)

# vamos gerar uma nova coluna em vip_scores_df com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_perm = {} # dicionário para mapear energia para zona espectral
for zone_name, start, end in spectral_cuts: # iterando sobre cada zona espectral (que tem nome, início e fim)
	for i in perm_df['energy']: # iterando sobre cada valor de energia no vip_scores_df
		i_float = float(i)
		if start <= i_float <= end:
			energy_to_zone_perm[i] = zone_name
perm_df['Zone'] = perm_df['energy'].map(energy_to_zone_perm) # a funcao map funciona como um buscador que substitui os valores de 'energy' pelos valores correspondentes no dicionário energy_to_zone_vip

# vamos filtrar vip_scores_df para manter apenas as zonas espectrais únicas com maior VIP score
perm_unique_df = perm_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
perm_unique_df = perm_unique_df.sort_values(by='Perm', ascending=False).reset_index(drop=True)
perm_unique_df


,energy,Perm,Zone
0,740,0.002448,700
1,750,0.002411,750
2,698,0.001969,650
3,1410,0.001873,1400
4,1382,0.001841,1350
5,1554,0.001822,1550
6,1548,0.001817,1500
7,1600,0.001804,1600
8,2206,0.001737,2200
9,1650,0.001723,1650


# **bagging - covariance**

In [ ]:
import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.5), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        mi_results_dict=all_results_cov[seed]['cov_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 109 | Descartados: 59
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 106 | Descartados: 62
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 104 | Descartados: 64
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 107 | Descartados: 61
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 105 | Descartados: 63
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 104 | Descartados: 64
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 105 | Descartados: 63
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 111 | Descartados: 57
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 106 | Descartados: 62


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2
0,1700 > -0.04,1500 > -0.04,1500 > -0.04
1,1900 > -0.04,1800 > -0.02,1300 > -0.01
2,800 <= -0.01,1400 > -0.01,1400 > -0.04
3,2000 > -0.04,1700 > -0.04,1500 > -0.02
4,1800 > 0.01,1600 > -0.04,1600 > -0.04
...,...,...,...
118,700 > -0.01,2100 <= -0.02,Class_A
119,1100 > 0.01,2400 <= -0.02,Class_B
120,700 <= -0.01,800 <= -0.01,NaN
121,Class_A,Class_A,NaN


In [11]:
from permutation import calculate_predicate_metrics_permutation

# LISTA DE SEMENTES A TESTAR
random_seeds = [0]

all_results_perm = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.5), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    perm_results_seed = calculate_predicate_metrics_permutation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        y_calclass=ycalclass,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        scoring='accuracy',
        task_type='classification',
        n_repeats=5,  # Usar 10-20 em produção para resultados mais estáveis
        random_state=0,
        n_jobs=33, # se usar 22 significa que vai usar 22 núcleos do processador em paralelo
        verbose=True,
        save_detailed_results=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    perm_results_seed_thresholded = {}
    for bag, df in perm_results_seed.items():
        # Verifica se é um DataFrame e se a coluna 'Permutation' existe
        if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
            filtered_df = df[df['Permutation'] > 0].copy()
            perm_results_seed_thresholded[bag] = filtered_df
        else:
            # Se não for DataFrame esperado, apenas copia
            perm_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_perm[seed] = {
        'bags_result': bags_result_seed,
        'perm_results_dict': perm_results_seed_thresholded
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_perm_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_perm[seed]['bags_result'],
        mi_results_dict=all_results_perm[seed]['perm_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_perm_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_perm_by_seed = {}
for seed in random_seeds:
    DG = graphs_perm_by_seed[seed]
    lrc_perm_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_perm_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_perm_by_seed[seed] = lrc_perm_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_perm_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_perm_df_seed = lrc_perm_by_seed[seed].rename(columns={'Node': f'Predicate_perm_Seed_{seed}'})
    lrc_perm_all_seeds_df = pd.concat([lrc_perm_all_seeds_df, lrc_perm_df_seed[[f'Predicate_perm_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_perm_unique_by_seed = {}
for seed, lrc_df in lrc_perm_by_seed.items():
    lrc_perm_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_perm_unique_df = lrc_perm_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_perm_unique_by_seed[seed] = lrc_perm_unique_df

lrc_perm_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 109 | Descartados: 59
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 106 | Descartados: 62
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 104 | Descartados: 64
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 107 | Descartados: 61
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 105 | Descartados: 63
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 104 | Descartados: 64
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 105 | Descartados: 63
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 101 | Descartados: 67
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 111 | Descartados: 57
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 106 | Descartados: 62


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_perm_Seed_0
0,1000 > -0.03
1,1300 <= 0.04
2,1600 > -0.04
3,1700 > -0.04
4,1700 <= 0.04
...,...
75,1400 > 0.01
76,1000 <= -0.01
77,800 <= -0.01
78,Class_A


In [11]:
all_results_perm[0]['perm_results_dict']['Bag_10']

,Predicate,Permutation
0,700 <= 0.03,0.093772
1,700 <= 0.01,0.076931
2,1500 > -0.04,0.065917
3,1500 <= 0.04,0.065281
4,700 > -0.02,0.064055
...,...,...
96,1100 <= 0.03,0.001475
97,1100 <= -0.01,0.001230
98,1100 <= 0.01,0.001228
99,1100 > -0.03,0.001146


# **Perturbation**

In [37]:
from permutation import calculate_predicate_metrics_permutation
import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='sum')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

# LISTA DE SEMENTES A TESTAR
random_seeds = [0]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = perm.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        perturbation_value=np.mean(y_predicted_numeric)*1000,
        metric='mean_relative_dev',   # Média com sinal (pode ser negativo)
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        mi_results_dict=all_results_pert[seed]['pert_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 630 | Descartados: 210
PERTURBATION IMPORTANCE PARA PREDICADOS
Valor de perturbação: 145.4545454545454
Métrica: mean_relative_dev
Total de folds: 10


[Bag_1] Processando 630 pr

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_pert_Seed_0
0,2380 <= 0.07
1,1540 > -0.05
2,1540 <= -0.05
3,2100 <= 0.07
4,1360 > -0.19
...,...
627,1760 <= -0.05
628,440 <= -0.07
629,440 <= 0.04
630,Class_A


In [8]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,700 > -0.12,7311.393654
1,700 > 0.06,6740.986302
2,750 > -0.12,6362.040985
3,750 > 0.07,6029.978795
4,700 <= 0.25,5634.802451
...,...,...
247,1150 <= 0.44,261.624487
248,2100 <= 0.17,245.696162
249,2200 <= -0.15,244.062280
250,2200 <= 0.44,242.394453


In [20]:
predicates_quantiles[0]['rule']

0      400 <= -0.27
1       400 > -0.27
2      400 <= -0.11
3       400 > -0.11
4       400 <= 0.07
           ...     
331    2450 > -0.16
332    2450 <= 0.18
333     2450 > 0.18
334    2450 <= 0.44
335     2450 > 0.44
Name: rule, Length: 336, dtype: object

In [22]:
from collections import defaultdict

# 1. Coletar posições de cada predicado em cada bag
positions_dict = defaultdict(list)

for bag_num in range(1, 11):
    bag_name = f'Bag_{bag_num}'
    bag_df = all_results_pert[0]['pert_results_dict'][bag_name]
    
    for position, predicate in enumerate(bag_df['Predicate'], start=1):
        positions_dict[predicate].append(position)

# 2. Calcular média e número de aparições
results = []
for predicate, positions in positions_dict.items():
    results.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone' : predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone'].values[0]
    })

# 3. Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_df = pd.DataFrame(results).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# 4. Lista final ordenada
lista_ordenada = ranking_df['Predicate'].tolist()
ranking_df

,Predicate,Mean_Position,Appearances,Zone
0,700 > -0.12,2.2,10,700
1,700 <= 0.25,3.6,10,700
2,700 > 0.06,3.7,10,700
3,750 > -0.12,5.7,10,750
4,750 > 0.07,6.8,10,750
...,...,...,...,...
247,2200 <= -0.15,246.3,10,2200
248,2200 > -0.15,246.9,10,2200
249,1150 <= 0.44,247.0,10,1150
250,2100 <= 0.17,247.5,10,2100


In [24]:
ranking_unique_df = ranking_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_unique_df['Zone']

0      700
1      750
2      550
3      650
4      800
5      500
6     1350
7     1550
8      950
9     1600
10    1000
11    1500
12    1400
13    1650
14    1450
15     900
16    1700
17    1050
18     850
19    1300
20    1750
21    1800
22    2150
23    1900
24    1250
25    2450
26    1100
27    2400
28    1950
29     600
30     450
31    1850
32    2000
33    1200
34     400
35    2050
36    2300
37    2250
38    2350
39    2100
40    1150
41    2200
Name: Zone, dtype: object

In [12]:
DG_fold = exp.build_fold_predicate_graph(
    bags_result=all_results_pert[0]['bags_result'],
    mi_results_dict=all_results_pert[0]['pert_results_dict'],
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência
    accumulate_cooccurrence_weights=False 
)

# Calcula LRC para cada nó e compõe DataFrame
lrc_df_fold = exp.calculate_lrc_single_graph(DG_fold, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df_fold = lrc_df_fold.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_df_fold

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: NÃO-ACUMULATIVA (peso fixo da matriz)

Folds processados: 10
Arestas criadas (antes de resolver bidirecionais): 2103

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 209
Critério de desempate: SCORE DE CONFIANÇA (soma dos ranks invertidos)
(Peso de co-ocorrência global é simétrico, então usamos score para decidir direção)

[700 > -0.12 ↔ 750 > 0.07]
  ✗ Removida: 700 > -0.12 → 750 > 0.07 (score=250.00, peso=110.00)
  ✓ Mantida:  750 > 0.07 → 700 > -0.12 (score=502.00, peso=110.00)

[700 > -0.12 ↔ 700 <= 0.25]
  ✗ Removida: 700 <= 0.25 → 700 > -0.12 (score=503.00, peso=115.00)
  ✓ Mantida:  700 > -0.12 → 700 <= 0.25 (score=504.00, peso=115.00)

[700 > 0.06 ↔ 700 <= 0.25]
  ✗ Removida: 700 <= 0.25 → 700 > 0.06

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,650 > 0.04,4.334175,650,0.04,>
1,650 > -0.11,4.251586,650,-0.11,>
2,1350 > -0.45,3.682559,1350,-0.45,>
3,500 > 0.06,3.679895,500,0.06,>
4,1750 > -0.47,3.648994,1750,-0.47,>
...,...,...,...,...,...
249,2350 > -0.15,0.934691,2350,-0.15,>
250,450 <= -0.16,0.922739,450,-0.16,<=
251,1150 <= -0.12,0.887386,1150,-0.12,<=
252,Class_A,0.000000,None,None,None


In [33]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    #len(lrc_cov_unique_df['Zone']),
    len(lrc_pert_unique_df['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Vip': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_coef': pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'].astype(str), max_len),
    'Ranking' : pad_list(ranking_unique_df['Zone'], max_len)
})

# for seed, lrc_unique_df in lrc_cov_unique_by_seed.items():
#     features_importance[f'LRC_cov_{seed}'] = pad_list(lrc_unique_df['Zone'].iloc[:10].tolist(), max_len)

# for seed, lrc_unique_df in lrc_pert_unique_by_seed.items():
#     features_importance[f'LRC_pert_{seed}'] = pad_list(lrc_unique_df['Zone'].iloc[:10].tolist(), max_len)

# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
#features_importance.to_excel('features_importance_soil_vnir.xlsx', index=False, sheet_name='Perm')
features_importance.head(20)

,Vip,Reg_coef,Shap,Ranking
0,700,700,700,700
1,750,750,500,750
2,650,650,650,550
3,800,550,550,650
4,600,500,1600,800
5,500,800,1350,500
6,550,1000,750,1350
7,450,950,1500,1550
8,400,1400,2450,950
9,850,1050,1400,1600


In [32]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if x is not None]
methods = ['Reg_coef', 'Shap', 'Ranking'] #+ [f'LRC_cov_{seed}' for seed in random_seeds] + [f'LRC_perm_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if x is not None]
    # Truncate both lists to the same length (minimum of both)
    min_len = min(len(reference_list), len(compare_list))
    ref_trunc = reference_list[:min_len]
    cmp_trunc = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trunc, cmp_trunc).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
#rbo_results.to_excel('rbo_soil_vnir.xlsx', index=False, sheet_name='perm')
rbo_results

,Reference,Method,RBO_Score
0,Vip,Reg_coef,0.886986
2,Vip,Ranking,0.852392
1,Vip,Shap,0.664785
